Part I: Data load, defining tiers for future test/train, and investigate distribution

In [3]:
import pandas as pd

#import the cleaned data
df=pd.read_csv('cleaned_apartment_data.csv')

#define pricing tiers
bins = [0,1000,2000,float('inf')]
labels = ["low", "medium", "high"]
df["price_category"] = pd.cut(df["price"], bins=bins, labels=labels, right=True)

#investigate the new tiers
df["price_category"].value_counts()

/var/folders/52/wpyglqgx0cg0m10bmt732_7c0000gn/T/ipykernel_9260/1835118211.py:4: DtypeWarning: Columns (0: address) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('cleaned_apartment_data.csv')


price_category
medium    57573
low       24200
high      17720
Name: count, dtype: int64

Part II: Define target and features, then split for train/test

In [15]:
#target and feature configuration
y = df["price_category"]

unused_cols = [
    "price", "price_display",
    "price_category",
    "title", "body", "address", "amenities", "source", "time",
    "currency", "price_type",
    "cityname",
]

X = df.drop(columns=unused_cols)

print(X.shape)
print(X.columns.tolist())
print(y.value_counts())

#split for train/test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print(y_train.value_counts(normalize=True).round(3))
print(y_test.value_counts(normalize=True).round(3))


(99493, 10)
['category', 'bathrooms', 'bedrooms', 'fee', 'has_photo', 'pets_allowed', 'square_feet', 'state', 'latitude', 'longitude']
price_category
medium    57573
low       24200
high      17720
Name: count, dtype: int64
price_category
medium    0.579
low       0.243
high      0.178
Name: proportion, dtype: float64
price_category
medium    0.579
low       0.243
high      0.178
Name: proportion, dtype: float64


Part 3: Column type conversion and baseline

In [19]:
#type conversion for categorical features
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

#make the test set have the same columns as the training set
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

#baseline: predict "medium" for every listing
from sklearn.metrics import classification_report, confusion_matrix

y_pred = ["medium"] * len(y_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[    0     0  3544]
 [    0     0  4840]
 [    0     0 11515]]
              precision    recall  f1-score   support

        high       0.00      0.00      0.00      3544
         low       0.00      0.00      0.00      4840
      medium       0.58      1.00      0.73     11515

    accuracy                           0.58     19899
   macro avg       0.19      0.33      0.24     19899
weighted avg       0.33      0.58      0.42     19899



/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

Part 4: Model A - logistic regression


In [20]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[2728  188  628]
 [ 217 3823  800]
 [2919 3217 5379]]
              precision    recall  f1-score   support

        high       0.47      0.77      0.58      3544
         low       0.53      0.79      0.63      4840
      medium       0.79      0.47      0.59     11515

    accuracy                           0.60     19899
   macro avg       0.59      0.68      0.60     19899
weighted avg       0.67      0.60      0.60     19899



/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Part 4: Model B - Random Forest

In [22]:
#random forest model setup
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=0,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 2843     8   693]
 [    9  3855   976]
 [  549   743 10223]]
              precision    recall  f1-score   support

        high       0.84      0.80      0.82      3544
         low       0.84      0.80      0.82      4840
      medium       0.86      0.89      0.87     11515

    accuracy                           0.85     19899
   macro avg       0.84      0.83      0.84     19899
weighted avg       0.85      0.85      0.85     19899



Part 5: Summarize the baseline, linear and random forest models

In [25]:
from sklearn.metrics import accuracy_score, f1_score

#predictions from each model
baseline_pred = ["medium"] * len(y_test)
logreg_pred = logreg.predict(X_test)
rf_pred = rf.predict(X_test)

#comparison table
results = pd.DataFrame({
    "Model": ["Baseline", "Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, logreg_pred),
        accuracy_score(y_test, rf_pred),
    ],
    "F1": [
        f1_score(y_test, baseline_pred, average="macro"),
        f1_score(y_test, logreg_pred, average="macro"),
        f1_score(y_test, rf_pred, average="macro"),
    ],
})

results.round(2)

,Model,Accuracy,F1
0,Baseline,0.58,0.24
1,Logistic Regression,0.60,0.60
2,Random Forest,0.85,0.84


Part 6: Select Random Forest and pull important features to summarize findings

In [26]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(10)

longitude      0.257118
square_feet    0.232809
latitude       0.214815
state_ca       0.049256
bathrooms      0.047784
bedrooms       0.031802
state_ma       0.020261
state_nj       0.012941
state_co       0.010706
state_md       0.009435
dtype: float64

Part 7: Summary of findings

Random forest is the clear winner 